### MFR Demo

In [1]:
import pandas as pd
from datasets import load_dataset
from utils import counting, mfr, evaluate

##
from train_module import train_and_eval
from torch.utils.data import DataLoader
import random
import numpy as np
import torch

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# Load data
dev_pub_data = load_dataset("weerayut/multilexnorm2026-dev-pub")

# Select data
lang = "en"
train, val = dev_pub_data["train"], dev_pub_data["validation"]

# # Smoke test baseline
# counts = counting(train)
# out = mfr(['bcause', 'u', 'r', 'funny'], counts)
# print("Smoke test:", out)

# ## Inference
# ds = pd.DataFrame(val)
# ds['pred'] = ds['raw'].apply(lambda x: mfr(x, counts))

# ## Evaluate 
# evaluate(ds['raw'].tolist(), ds['norm'].tolist(), ds['pred'].tolist())

c:\Users\DH\Desktop\플래너\5. 학기 자료\26-1\2.인지개\과제\MultiLexNorm2026-main\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Submission demo

In [2]:
#data, save_path = load_dataset("weerayut/multilexnorm2026-dev-pub"), "outputs/submission_dev"
# data, save_path = load_dataset("weerayut/multilexnorm2026-full-pub"), "outputs/submission_full"

In [3]:
# def prediction(train, test):
#     train_df = train.to_pandas()
#     test_df  = test.to_pandas()

#     # Build a per-language dictionary
#     count_langs = {} 
#     for lang in train_df["lang"].unique():
#         train_lang = train_df.loc[train_df["lang"] == lang]
#         count_langs[lang] = counting(train_lang.to_dict(orient="records"))

#     # Prediction per language
#     test_df["pred"] = test_df.apply(
#         lambda r: mfr(r["raw"], count_langs.get(r["lang"])),
#         axis=1
#     )
#     return test_df

In [4]:
from datasets import concatenate_datasets

## predict and save dev
# train = concatenate_datasets([data['train'], data['validation']])
# out = prediction(train, data['test'])
# out.to_json(f"{save_path}/predictions.json", orient="records")

In [5]:
# out[['raw', 'pred', 'lang']].head(3)

### Zip files

In [6]:
# from utils import zip_files_flat

# zip_files_flat(save_path, f"{save_path}.zip")

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

c:\Users\DH\Desktop\플래너\5. 학기 자료\26-1\2.인지개\과제\MultiLexNorm2026-main\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
def make_pairs(dataset, num_neg=1, seed=42):
    rng = random.Random(seed)  # 🔥 핵심

    pairs = []
    norms = [" ".join(x['norm']) for x in dataset]

    for item in dataset:
        raw = " ".join(item['raw'])
        norm = " ".join(item['norm'])

        pairs.append((raw, norm, 1))

        for _ in range(num_neg):
            while True:
                neg = rng.choice(norms)
                if neg != norm:
                    pairs.append((raw, neg, 0))
                    break

    return pairs

In [9]:
from torch.utils.data import Dataset

class PairDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_len=64):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        text1, text2, label = self.pairs[idx]

        # 🔥 여기 추가
        text1 = " ".join(text1) if isinstance(text1, list) else text1
        text2 = " ".join(text2) if isinstance(text2, list) else text2

        enc1 = self.tokenizer(
            text1,
            truncation=True,
            padding="max_length",
            max_length=64,
            return_tensors="pt"
        )

        enc2 = self.tokenizer(
            text2,
            truncation=True,
            padding="max_length",
            max_length=64,
            return_tensors="pt"
        )

        return {
            "input_ids1": enc1["input_ids"].squeeze(0),
            "attention_mask1": enc1["attention_mask"].squeeze(0),
            "input_ids2": enc2["input_ids"].squeeze(0),
            "attention_mask2": enc2["attention_mask"].squeeze(0),
            "labels": label
        }

In [10]:
print(type(train))
print(len(train))
print(train[0])
print(train.column_names)

<class 'datasets.arrow_dataset.Dataset'>
39178
{'raw': ['Dette', 'er', 'ikke', 'tilfaeldigt', '.'], 'norm': ['Dette', 'er', 'ikke', 'tilfældigt', '.'], 'lang': 'da'}
['raw', 'norm', 'lang']


In [11]:
print(train[0]['raw'])
print(type(train[0]['raw']))

print(train[0]['norm'])
print(type(train[0]['norm']))

['Dette', 'er', 'ikke', 'tilfaeldigt', '.']
<class 'list'>
['Dette', 'er', 'ikke', 'tilfældigt', '.']
<class 'list'>


In [12]:
pairs_train = make_pairs(train, seed=42)
pairs_val = make_pairs(val, seed=42)

train_dataset = PairDataset(pairs_train, tokenizer)
val_dataset = PairDataset(pairs_val, tokenizer)

g = torch.Generator()
g.manual_seed(42)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0,generator=g)
val_loader = DataLoader(val_dataset, batch_size=16, num_workers=0)

model, eer = train_and_eval(train_loader, val_loader)

🔥 checkpoint 로드: epoch 1부터 시작


Epoch 1: 100%|██████████| 4898/4898 [2:30:55<00:00,  1.85s/it]  


[Epoch 1] loss: 0.0171


Epoch 2: 100%|██████████| 4898/4898 [2:32:48<00:00,  1.87s/it]  


[Epoch 2] loss: 0.0086


Evaluating: 100%|██████████| 1051/1051 [09:53<00:00,  1.77it/s]


🔥 REAL EER: 0.0061 (threshold=0.7492)
